# MedicalNLP-RAG — Phase 6
## Final Analysis & Manuscript Package

This notebook is a **read-only synthesis phase**. It does not train, tune, rerank, regenerate,
recalibrate, or modify any frozen model or policy.

### Purpose

- verify the integrity of Phase 3C–5 frozen outputs;
- consolidate validation and one-time locked-test results;
- audit evaluation consistency, especially the list-question evaluator;
- create manuscript-ready tables and figures;
- create supplementary tables;
- create a single source of truth for manuscript numbers;
- document that no post-test tuning was performed.

### Critical rule

The Phase 5 locked internal test has already been evaluated once.  
**This notebook must never reopen the locked test labels or rerun the model pipeline.**

All locked-test results are read only from the frozen Phase 5 outputs.

## 0. Configuration

In [ ]:
PROJECT_NAME = "MedicalNLP_RAG"
SEED = 42

# Read-only synthesis flags
ALLOW_POST_TEST_MODEL_TUNING = False
ALLOW_LOCKED_TEST_RAW_LABEL_ACCESS = False
ALLOW_PHASE5_RERUN = False

# Manuscript output conventions
ROUND_MAIN = 4
ROUND_SUPP = 6
FIG_DPI = 300

# Frozen task order
TASKS = ["yesno", "factoid", "list", "summary"]

PRIMARY_POLICY = "conservative_summary80"
SECONDARY_POLICY = "expanded_summary_factoid80"

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Imports and output directories

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import OrderedDict
import hashlib, json, math, os, re, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")

ROOT = Path(f"/content/drive/MyDrive/{PROJECT_NAME}")

PHASE3B_DIR = ROOT / "results" / "phase3B_retrieval"
PHASE3C_DIR = ROOT / "results" / "phase3C_reranking"
PHASE4A_DIR = ROOT / "results" / "phase4A_generation"
PHASE4B_DIR = ROOT / "results" / "phase4B_evidence_filtering"
PHASE4C_DIR = ROOT / "results" / "phase4C_grounding_v2"
PHASE4D_DIR = ROOT / "results" / "phase4D_v2_task_specific_confidence"
PHASE4E_DIR = ROOT / "results" / "phase4E_task_specific_selective_answering"
PHASE5_DIR = ROOT / "results" / "phase5_one_time_locked_internal_test"

PHASE6_DIR = ROOT / "results" / "phase6_final_analysis_manuscript_package"
TABLE_DIR = PHASE6_DIR / "tables"
FIG_DIR = PHASE6_DIR / "figures"
SUPP_DIR = PHASE6_DIR / "supplementary"
AUDIT_DIR = PHASE6_DIR / "audit"
EXPORT_DIR = PHASE6_DIR / "exports"

for p in [PHASE6_DIR, TABLE_DIR, FIG_DIR, SUPP_DIR, AUDIT_DIR, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Phase 6 output:", PHASE6_DIR)

## 3. General helpers

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def first_existing(paths, required=True):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    if required:
        raise FileNotFoundError("None of these files exist:\n" + "\n".join(map(str, paths)))
    return None

def read_table(path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".json", ".jsn"}:
        obj = load_json(path)
        if isinstance(obj, list):
            return pd.DataFrame(obj)
        return pd.json_normalize(obj)
    raise ValueError(path)

def flat_metric_rows(d, prefix=""):
    rows = []
    if isinstance(d, dict):
        for k, v in d.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            if isinstance(v, dict):
                rows.extend(flat_metric_rows(v, key))
            elif isinstance(v, list):
                continue
            else:
                rows.append((key, v))
    return rows

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def write_df_sheet(ws, df, freeze_panes="A2"):
    ws.append(list(df.columns))
    for c in ws[1]:
        c.font = Font(bold=True)
        c.alignment = Alignment(horizontal="center", vertical="center")
    for row in df.itertuples(index=False, name=None):
        ws.append(list(row))
    ws.freeze_panes = freeze_panes
    for idx, col in enumerate(df.columns, start=1):
        values = [str(col)] + [str(v) for v in df[col].head(500).tolist()]
        width = min(max(max(len(v) for v in values) + 2, 10), 45)
        ws.column_dimensions[get_column_letter(idx)].width = width

def save_workbook(sheets, path):
    wb = Workbook()
    wb.remove(wb.active)
    for name, df in sheets.items():
        ws = wb.create_sheet(title=str(name)[:31])
        write_df_sheet(ws, df)
    wb.save(path)

print("Helpers ready.")

# Part I — Immutable audit

## 4. Locate all freeze / final-report artifacts

In [ ]:
FROZEN_RETRIEVAL_JSON = first_existing([
    PHASE3C_DIR / "frozen_retrieval_pipeline.json",
])

FROZEN_EVIDENCE_JSON = first_existing([
    PHASE4B_DIR / "frozen_evidence_strategy.json",
])

FROZEN_GROUNDING_JSON = first_existing([
    PHASE4C_DIR / "frozen_grounding_method_v2.json",
])

FROZEN_CONFIDENCE_JSON = first_existing([
    PHASE4D_DIR / "frozen_confidence_method_v2.json",
])

FROZEN_POLICY_JSON = first_existing([
    PHASE4E_DIR / "frozen_selective_answering_policy.json",
])

PHASE5_FINAL_REPORT = first_existing([
    PHASE5_DIR / "phase5_final_locked_test_report.json",
])

PHASE5_MARKER = first_existing([
    PHASE5_DIR / "one_time_evaluation_marker.json",
])

PHASE5_STAGEA_MANIFEST = first_existing([
    PHASE5_DIR / "stageA_label_blind" / "stageA_freeze_manifest.json",
])

artifact_paths = OrderedDict([
    ("phase3C_retrieval_freeze", FROZEN_RETRIEVAL_JSON),
    ("phase4B_evidence_freeze", FROZEN_EVIDENCE_JSON),
    ("phase4C_grounding_freeze", FROZEN_GROUNDING_JSON),
    ("phase4D_confidence_freeze", FROZEN_CONFIDENCE_JSON),
    ("phase4E_policy_freeze", FROZEN_POLICY_JSON),
    ("phase5_stageA_manifest", PHASE5_STAGEA_MANIFEST),
    ("phase5_final_report", PHASE5_FINAL_REPORT),
    ("phase5_one_time_marker", PHASE5_MARKER),
])

artifact_inventory = pd.DataFrame([
    {
        "artifact": k,
        "path": str(p),
        "exists": p.exists(),
        "sha256": sha256_file(p) if p.exists() else None,
        "size_bytes": p.stat().st_size if p.exists() else None,
    }
    for k, p in artifact_paths.items()
])

display(artifact_inventory)
artifact_inventory.to_csv(AUDIT_DIR / "artifact_inventory_sha256.csv", index=False)

## 5. Final one-time-test integrity assertions

In [ ]:
assert ALLOW_POST_TEST_MODEL_TUNING is False
assert ALLOW_LOCKED_TEST_RAW_LABEL_ACCESS is False
assert ALLOW_PHASE5_RERUN is False

retrieval_freeze = load_json(FROZEN_RETRIEVAL_JSON)
evidence_freeze = load_json(FROZEN_EVIDENCE_JSON)
grounding_freeze = load_json(FROZEN_GROUNDING_JSON)
confidence_freeze = load_json(FROZEN_CONFIDENCE_JSON)
policy_freeze = load_json(FROZEN_POLICY_JSON)
phase5 = load_json(PHASE5_FINAL_REPORT)
marker = load_json(PHASE5_MARKER)
stagea_manifest = load_json(PHASE5_STAGEA_MANIFEST)

assert marker["status"] == "COMPLETE"
assert phase5["post_hoc_test_tuning_performed"] is False
assert phase5["locked_test_used_for_model_selection"] is False
assert phase5["locked_test_used_for_coverage_selection"] is False
assert phase5["locked_test_used_for_threshold_tuning"] is False

assert policy_freeze["internal_test_evaluated"] is False
assert policy_freeze["primary_policy"]["name"] == PRIMARY_POLICY
assert policy_freeze["secondary_policy"]["name"] == SECONDARY_POLICY

assert phase5["primary_policy"]["policy"] == PRIMARY_POLICY
assert phase5["secondary_policy"]["policy"] == SECONDARY_POLICY

assert phase5["locked_test_questions"] == 860
assert stagea_manifest["n_locked_test"] == 860

expected_final_hash = marker["final_report_sha256"]
actual_final_hash = sha256_file(PHASE5_FINAL_REPORT)
assert expected_final_hash == actual_final_hash

audit_summary = {
    "phase5_status": marker["status"],
    "phase5_final_report_sha256": actual_final_hash,
    "stageA_manifest_sha256_current": sha256_file(PHASE5_STAGEA_MANIFEST),
    "locked_test_questions": phase5["locked_test_questions"],
    "post_hoc_test_tuning_performed": phase5["post_hoc_test_tuning_performed"],
    "locked_test_used_for_model_selection": phase5["locked_test_used_for_model_selection"],
    "locked_test_used_for_coverage_selection": phase5["locked_test_used_for_coverage_selection"],
    "locked_test_used_for_threshold_tuning": phase5["locked_test_used_for_threshold_tuning"],
}
save_json(audit_summary, AUDIT_DIR / "final_integrity_audit.json")
print(json.dumps(audit_summary, indent=2))
print("PASS — Phase 5 completion and no-post-test-tuning assertions verified.")

### Important audit note

The Stage A manifest was intentionally updated after Stage B to record that gold scoring had
occurred, so its current SHA256 need not equal the pre-gold Stage A SHA printed before Stage B.
The immutable chain is preserved by the Phase 5 final report, completion marker, and the stored
artifact hashes.

# Part II — Load canonical final results

## 6. Load Phase 5 canonical locked-test tables

In [ ]:
P5_TABLE_DIR = PHASE5_DIR / "tables"

P5_ANSWER = first_existing([P5_TABLE_DIR / "locked_test_answer_summary.csv"])
P5_RETRIEVAL = first_existing([P5_TABLE_DIR / "locked_test_retrieval_per_question.csv"])
P5_GROUNDING = first_existing([P5_TABLE_DIR / "locked_test_grounding_summary.csv"])
P5_CONF = first_existing([P5_TABLE_DIR / "locked_test_confidence_metrics_by_task.csv"])
P5_POLICY_TASK = first_existing([P5_TABLE_DIR / "locked_test_policy_task_results.csv"])
P5_POLICY_SUMMARY = first_existing([P5_TABLE_DIR / "locked_test_policy_summary.csv"])
P5_POLICY_BOOT = first_existing([P5_TABLE_DIR / "locked_test_policy_macro_gain_bootstrap.csv"])
P5_VAL_TEST = first_existing([P5_TABLE_DIR / "validation_vs_locked_test_comparison.csv"])

answer_test = pd.read_csv(P5_ANSWER)
retrieval_test_perq = pd.read_csv(P5_RETRIEVAL)
grounding_test = pd.read_csv(P5_GROUNDING)
confidence_test = pd.read_csv(P5_CONF)
policy_task_test = pd.read_csv(P5_POLICY_TASK)
policy_summary_test = pd.read_csv(P5_POLICY_SUMMARY)
policy_boot_test = pd.read_csv(P5_POLICY_BOOT)
val_test_comparison = pd.read_csv(P5_VAL_TEST)

print("Phase 5 canonical tables loaded.")
display(answer_test.T)

## 7. Load Phase 4 validation reference tables

In [ ]:
P4E_REPORT = first_existing([
    PHASE4E_DIR / "phase4E_report.json",
], required=False)

P4E_POLICY_SUMMARY = first_existing([
    PHASE4E_DIR / "tables" / "policy_summary.csv",
    PHASE4E_DIR / "tables" / "frozen_policy_summary.csv",
    PHASE4E_DIR / "policy_summary.csv",
], required=False)

P4E_BOOT = first_existing([
    PHASE4E_DIR / "tables" / "policy_macro_gain_bootstrap.csv",
    PHASE4E_DIR / "tables" / "policy_bootstrap.csv",
    PHASE4E_DIR / "policy_macro_gain_bootstrap.csv",
], required=False)

P4D_METRICS = first_existing([
    PHASE4D_DIR / "tables" / "task_specific_confidence_metrics.csv",
    PHASE4D_DIR / "task_specific_confidence_metrics.csv",
], required=False)

P4C_REPORT = first_existing([
    PHASE4C_DIR / "phase4C_v2_report.json",
    PHASE4C_DIR / "phase4C_report.json",
], required=False)

P4B_REPORT = first_existing([
    PHASE4B_DIR / "phase4B_report.json",
    PHASE4B_DIR / "phase4B_evidence_filtering_report.json",
], required=False)

print("Optional validation source discovery:")
for name, p in {
    "P4E_REPORT": P4E_REPORT,
    "P4E_POLICY_SUMMARY": P4E_POLICY_SUMMARY,
    "P4E_BOOT": P4E_BOOT,
    "P4D_METRICS": P4D_METRICS,
    "P4C_REPORT": P4C_REPORT,
    "P4B_REPORT": P4B_REPORT,
}.items():
    print(f"{name}: {p}")

## 8. Canonical validation numbers

The frozen Phase 4E policy file is the source of truth for validation policy results.
Task-level validation answer scores use the frozen `document_top5` selection from Phase 4B.

In [ ]:
validation_answer = {
    "n": 862,
    "json_valid_rate": 1.0,
    "rouge1_f": 0.354790,
    "rouge2_f": 0.161048,
    "rougeL_f": 0.260493,
    "yesno_accuracy": 0.913793,
    "yesno_macro_f1": 0.887597,
    "factoid_strict": 0.411765,
    "factoid_lenient": 0.501961,
    "factoid_mrr": 0.447908,
    "list_precision": 0.517776,
    "list_recall": 0.469683,
    "list_f1": 0.465642,
    "summary_rougeL_f": 0.242859,
    "macro_task_score": 0.517551,
}

validation_retrieval = {
    "recall@10": 0.694066,
    "MAP@10": 0.740771,
    "MRR@10": 0.867879,
    "nDCG@10": 0.807320,
}

validation_policy_a = policy_freeze["policy_a_validation_summary"]
validation_policy_b = policy_freeze["policy_b_validation_summary"]

validation_policy_bootstrap = pd.DataFrame([
    {
        "policy": PRIMARY_POLICY,
        "macro_gain_mean": 0.0020867722738014604,
        "ci95_low": 0.0006945255059218475,
        "ci95_high": 0.003583906407047535,
        "bootstrap_probability_gain_gt_0": 0.9984,
    },
    {
        "policy": SECONDARY_POLICY,
        "macro_gain_mean": 0.011143219355563014,
        "ci95_low": 0.003967486107535817,
        "ci95_high": 0.01867363949345832,
        "bootstrap_probability_gain_gt_0": 0.9998,
    },
])

print("Validation reference prepared.")

# Part III — Master results

## 9. Master answer-quality table: validation vs locked test

In [ ]:
metric_labels = OrderedDict([
    ("json_valid_rate", "JSON valid rate"),
    ("rouge1_f", "ROUGE-1 F"),
    ("rouge2_f", "ROUGE-2 F"),
    ("rougeL_f", "ROUGE-L F"),
    ("yesno_accuracy", "Yes/no accuracy"),
    ("yesno_macro_f1", "Yes/no Macro-F1"),
    ("factoid_strict", "Factoid strict accuracy"),
    ("factoid_lenient", "Factoid lenient accuracy"),
    ("factoid_mrr", "Factoid MRR"),
    ("list_precision", "List precision"),
    ("list_recall", "List recall"),
    ("list_f1", "List F1"),
    ("summary_rougeL_f", "Summary ROUGE-L F"),
    ("macro_task_score", "MacroTaskScore"),
])

test_answer_dict = answer_test.iloc[0].to_dict()

answer_master_rows = []
for key, label in metric_labels.items():
    v = safe_float(validation_answer.get(key))
    t = safe_float(test_answer_dict.get(key))
    answer_master_rows.append({
        "metric_key": key,
        "metric": label,
        "validation": v,
        "locked_test": t,
        "test_minus_validation": t - v,
        "relative_change_vs_validation": (t - v) / v if v not in [0, np.nan] else np.nan,
    })

answer_master = pd.DataFrame(answer_master_rows)
answer_master.to_csv(TABLE_DIR / "Table_answer_quality_validation_vs_locked_test.csv", index=False)
display(answer_master.round(6))

## 10. Quantify which task drives MacroTaskScore transport

In [ ]:
task_transport = pd.DataFrame([
    {
        "task": "yesno",
        "validation_task_score": validation_answer["yesno_accuracy"],
        "locked_test_task_score": test_answer_dict["yesno_accuracy"],
    },
    {
        "task": "factoid",
        "validation_task_score": validation_answer["factoid_mrr"],
        "locked_test_task_score": test_answer_dict["factoid_mrr"],
    },
    {
        "task": "list",
        "validation_task_score": validation_answer["list_f1"],
        "locked_test_task_score": test_answer_dict["list_f1"],
    },
    {
        "task": "summary",
        "validation_task_score": validation_answer["summary_rougeL_f"],
        "locked_test_task_score": test_answer_dict["summary_rougeL_f"],
    },
])

task_transport["delta"] = (
    task_transport["locked_test_task_score"] - task_transport["validation_task_score"]
)
task_transport["contribution_to_macro_delta"] = task_transport["delta"] / 4.0

macro_delta_from_tasks = float(task_transport["contribution_to_macro_delta"].sum())
macro_delta_direct = (
    float(test_answer_dict["macro_task_score"]) - validation_answer["macro_task_score"]
)
assert abs(macro_delta_from_tasks - macro_delta_direct) < 5e-6

task_transport["share_of_absolute_macro_drop"] = (
    task_transport["contribution_to_macro_delta"].abs()
    / abs(macro_delta_direct)
)

task_transport.to_csv(TABLE_DIR / "Table_task_contribution_to_macro_transport.csv", index=False)
display(task_transport.round(6))
print("Macro delta:", macro_delta_direct)

## 11. Master retrieval table

In [ ]:
retrieval_test_summary = phase5["retrieval_locked_test"]

retrieval_master = pd.DataFrame([
    {
        "metric": metric,
        "validation": validation_retrieval[metric],
        "locked_test": safe_float(retrieval_test_summary[metric]),
        "test_minus_validation": safe_float(retrieval_test_summary[metric]) - validation_retrieval[metric],
    }
    for metric in ["recall@10", "MAP@10", "MRR@10", "nDCG@10"]
])

retrieval_master.to_csv(TABLE_DIR / "Table_retrieval_validation_vs_locked_test.csv", index=False)
display(retrieval_master.round(6))

## 12. Master policy table

In [ ]:
validation_policy_summary = pd.DataFrame([
    {
        "policy": validation_policy_a["policy"],
        "n_total": validation_policy_a["n_total"],
        "n_answered": validation_policy_a["n_answered"],
        "n_abstained": validation_policy_a["n_abstained"],
        "coverage": validation_policy_a["overall_question_coverage"],
        "abstention_rate": validation_policy_a["overall_abstention_rate"],
        "full_macro_task_score": validation_policy_a["full_macro_task_score"],
        "selective_macro_task_score": validation_policy_a["selective_macro_task_score"],
        "macro_task_score_gain": validation_policy_a["macro_task_score_gain"],
        "split": "validation",
    },
    {
        "policy": validation_policy_b["policy"],
        "n_total": validation_policy_b["n_total"],
        "n_answered": validation_policy_b["n_answered"],
        "n_abstained": validation_policy_b["n_abstained"],
        "coverage": validation_policy_b["overall_question_coverage"],
        "abstention_rate": validation_policy_b["overall_abstention_rate"],
        "full_macro_task_score": validation_policy_b["full_macro_task_score"],
        "selective_macro_task_score": validation_policy_b["selective_macro_task_score"],
        "macro_task_score_gain": validation_policy_b["macro_task_score_gain"],
        "split": "validation",
    },
])

test_policy_summary = policy_summary_test.rename(
    columns={
        "overall_question_coverage": "coverage",
        "overall_abstention_rate": "abstention_rate",
    }
).copy()
test_policy_summary["split"] = "locked_test"

policy_master = pd.concat(
    [validation_policy_summary, test_policy_summary[validation_policy_summary.columns]],
    ignore_index=True,
)

policy_master.to_csv(TABLE_DIR / "Table_selective_policy_validation_vs_locked_test.csv", index=False)
display(policy_master.round(6))

## 13. Policy bootstrap transport table

In [ ]:
test_boot = policy_boot_test.rename(columns={
    "mean_gain": "macro_gain_mean",
    "ci95_low": "ci95_low",
    "ci95_high": "ci95_high",
}).copy()

validation_boot = validation_policy_bootstrap.copy()
validation_boot["split"] = "validation"
test_boot["split"] = "locked_test"

boot_cols = [
    "policy", "split", "macro_gain_mean",
    "ci95_low", "ci95_high", "bootstrap_probability_gain_gt_0"
]
policy_bootstrap_master = pd.concat([
    validation_boot[boot_cols],
    test_boot[boot_cols],
], ignore_index=True)

policy_bootstrap_master.to_csv(
    TABLE_DIR / "Table_policy_bootstrap_validation_vs_locked_test.csv", index=False
)
display(policy_bootstrap_master.round(6))

# Part IV — Confidence and grounding transport

## 14. Locked-test confidence results, with frozen validation context

In [ ]:
validation_confidence_context = pd.DataFrame([
    {"task":"yesno",   "raw_spearman_rho":-0.042659, "raw_spearman_p":0.517928, "isotonic_mae":0.158701, "isotonic_rmse":0.281046, "isotonic_ece":0.013544},
    {"task":"factoid", "raw_spearman_rho": 0.187837, "raw_spearman_p":0.002598, "isotonic_mae":0.457079, "isotonic_rmse":0.478493, "isotonic_ece":0.077732},
    {"task":"list",    "raw_spearman_rho":-0.006589, "raw_spearman_p":0.932041, "isotonic_mae":0.335664, "isotonic_rmse":0.381237, "isotonic_ece":0.051496},
    {"task":"summary", "raw_spearman_rho": 0.325853, "raw_spearman_p":0.000002, "isotonic_mae":0.069319, "isotonic_rmse":0.095494, "isotonic_ece":0.002597},
])

raw_test = confidence_test[confidence_test["method"] == "task_ridge_raw"][
    ["task","spearman_rho","spearman_p"]
].rename(columns={
    "spearman_rho":"locked_test_raw_spearman_rho",
    "spearman_p":"locked_test_raw_spearman_p"
})

iso_test = confidence_test[confidence_test["method"] == "task_ridge_isotonic"][
    ["task","mae","rmse","ece_10bin","calibration_bias",
     "calibration_slope","mean_confidence","mean_observed_task_score"]
].rename(columns={
    "mae":"locked_test_isotonic_mae",
    "rmse":"locked_test_isotonic_rmse",
    "ece_10bin":"locked_test_isotonic_ece",
})

confidence_transport = (
    validation_confidence_context
    .merge(raw_test, on="task", validate="one_to_one")
    .merge(iso_test, on="task", validate="one_to_one")
)

confidence_transport["rho_delta_test_minus_validation"] = (
    confidence_transport["locked_test_raw_spearman_rho"]
    - confidence_transport["raw_spearman_rho"]
)

confidence_transport.to_csv(TABLE_DIR / "Table_confidence_transport.csv", index=False)
display(confidence_transport.round(6))

### Interpretation lock

A positive locked-test correlation for a task that was validation-ineligible **does not modify**
the frozen eligibility decision. In particular, list questions remain excluded from the
prespecified selective policies. Any discussion of the locked-test list correlation is
post-hoc/exploratory only.

## 15. Locked-test grounding table

In [ ]:
grounding_test.to_csv(TABLE_DIR / "Table_locked_test_grounding.csv", index=False)
display(grounding_test.round(6))

# Part V — List evaluator consistency audit

## 16. Evaluation-consistency audit plan

The locked-test list F1 decreased substantially relative to validation. Before manuscript
submission, we verify that this is not caused by an accidental evaluator change.

This audit is **not allowed to change the frozen test pipeline**. It only checks:

1. whether the Phase 5 list scorer matches the canonical BioASQ-style normalized matching logic
   used in development;
2. whether the validation score stored in Phase 4B is internally consistent with its own
   precision/recall/F1 triplet;
3. whether the locked-test score is internally consistent with its own precision/recall/F1 triplet;
4. whether the arithmetic definition of F1 is identical.

If raw Phase 4B predictions are discoverable, the notebook optionally records their path for a
future code-level reproducibility audit, but it does not rescore or replace frozen values.

In [ ]:
def harmonic_f1(p, r):
    return 2*p*r/(p+r) if p+r > 0 else 0.0

val_p = validation_answer["list_precision"]
val_r = validation_answer["list_recall"]
val_f1_macro_question = validation_answer["list_f1"]

test_p = safe_float(test_answer_dict["list_precision"])
test_r = safe_float(test_answer_dict["list_recall"])
test_f1_macro_question = safe_float(test_answer_dict["list_f1"])

audit_rows = [
    {
        "split": "validation",
        "reported_precision": val_p,
        "reported_recall": val_r,
        "reported_mean_per_question_f1": val_f1_macro_question,
        "harmonic_of_aggregate_precision_recall": harmonic_f1(val_p, val_r),
        "note": "Mean per-question F1 need not equal harmonic mean of aggregate P/R.",
    },
    {
        "split": "locked_test",
        "reported_precision": test_p,
        "reported_recall": test_r,
        "reported_mean_per_question_f1": test_f1_macro_question,
        "harmonic_of_aggregate_precision_recall": harmonic_f1(test_p, test_r),
        "note": "Mean per-question F1 need not equal harmonic mean of aggregate P/R.",
    },
]
list_metric_audit = pd.DataFrame(audit_rows)
display(list_metric_audit.round(6))

# Discover likely Phase 4B raw generation/result files without reading locked-test labels.
candidate_files = sorted([
    p for p in PHASE4B_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".csv",".json",".jsonl"}
    and any(tok in p.name.lower() for tok in ["document_top5","generation","prediction","score","result"])
])

discovery = pd.DataFrame({
    "candidate_phase4B_file": [str(p) for p in candidate_files[:100]],
    "size_bytes": [p.stat().st_size for p in candidate_files[:100]],
})
discovery.to_csv(AUDIT_DIR / "phase4B_list_evaluator_candidate_files.csv", index=False)

list_audit_conclusion = {
    "validation_list_f1": val_f1_macro_question,
    "locked_test_list_f1": test_f1_macro_question,
    "absolute_delta": test_f1_macro_question - val_f1_macro_question,
    "phase6_changed_any_score": False,
    "phase6_rescored_locked_test": False,
    "phase6_used_locked_test_for_tuning": False,
    "note": (
        "The observed validation-to-test drop is retained as a genuine transport result unless "
        "a separate reproducibility audit finds an implementation mismatch. Phase 6 does not "
        "replace any frozen score."
    ),
}
save_json(list_audit_conclusion, AUDIT_DIR / "list_evaluator_consistency_audit.json")
print(json.dumps(list_audit_conclusion, indent=2))
print("Candidate Phase 4B files discovered:", len(candidate_files))

# Part VI — Manuscript-ready figures

## 17. Figure 1 — Validation vs locked-test task scores

In [ ]:
task_fig = task_transport.copy()
task_fig["task_label"] = task_fig["task"].map({
    "yesno": "Yes/no",
    "factoid": "Factoid",
    "list": "List",
    "summary": "Summary",
})

x = np.arange(len(task_fig))
w = 0.36

fig, ax = plt.subplots(figsize=(8.2, 5.0))
ax.bar(x - w/2, task_fig["validation_task_score"], width=w, label="Validation")
ax.bar(x + w/2, task_fig["locked_test_task_score"], width=w, label="Locked test")
ax.set_xticks(x)
ax.set_xticklabels(task_fig["task_label"])
ax.set_ylim(0, 1)
ax.set_ylabel("Task-specific benchmark score")
ax.set_title("Task performance: validation vs one-time locked test")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "Figure1_task_performance_validation_vs_locked_test.png",
            dpi=FIG_DPI, bbox_inches="tight")
plt.show()

## 18. Figure 2 — Retrieval transport

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 4.8))
x = np.arange(len(retrieval_master))
w = 0.36
ax.bar(x - w/2, retrieval_master["validation"], width=w, label="Validation")
ax.bar(x + w/2, retrieval_master["locked_test"], width=w, label="Locked test")
ax.set_xticks(x)
ax.set_xticklabels(retrieval_master["metric"])
ax.set_ylim(0, 1)
ax.set_ylabel("Retrieval metric")
ax.set_title("Frozen retrieval pipeline transport")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "Figure2_retrieval_validation_vs_locked_test.png",
            dpi=FIG_DPI, bbox_inches="tight")
plt.show()

## 19. Figure 3 — Selective policy gains only

In [ ]:
gain_plot = policy_master.pivot(index="policy", columns="split", values="macro_task_score_gain")
gain_plot = gain_plot.loc[[PRIMARY_POLICY, SECONDARY_POLICY]]

fig, ax = plt.subplots(figsize=(8.4, 5.0))
x = np.arange(len(gain_plot))
w = 0.36
ax.bar(x - w/2, gain_plot["validation"], width=w, label="Validation")
ax.bar(x + w/2, gain_plot["locked_test"], width=w, label="Locked test")
ax.set_xticks(x)
ax.set_xticklabels(["Primary: Summary-80", "Secondary: Summary+Factoid-80"])
ax.set_ylabel("MacroTaskScore gain")
ax.set_title("Prespecified selective-answering gain transport")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "Figure3_selective_policy_gain_transport.png",
            dpi=FIG_DPI, bbox_inches="tight")
plt.show()

## 20. Figure 4 — Coverage vs selective MacroTaskScore

In [ ]:
policy_plot = policy_master.copy()
policy_plot["label"] = policy_plot["split"] + " | " + policy_plot["policy"]

fig, ax = plt.subplots(figsize=(8.0, 5.2))
for split in ["validation", "locked_test"]:
    g = policy_plot[policy_plot["split"] == split]
    ax.scatter(g["coverage"], g["selective_macro_task_score"], s=70, label=split)
    for row in g.itertuples(index=False):
        short = "A" if row.policy == PRIMARY_POLICY else "B"
        ax.annotate(short, (row.coverage, row.selective_macro_task_score),
                    textcoords="offset points", xytext=(5,5))
ax.set_xlabel("Overall question coverage")
ax.set_ylabel("Selective MacroTaskScore")
ax.set_title("Coverage–performance trade-off of frozen policies")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "Figure4_coverage_performance_tradeoff.png",
            dpi=FIG_DPI, bbox_inches="tight")
plt.show()

## 21. Figure 5 — Confidence discrimination transport

In [ ]:
rho_plot = confidence_transport[[
    "task","raw_spearman_rho","locked_test_raw_spearman_rho"
]].copy()
rho_plot["task_label"] = rho_plot["task"].map({
    "yesno":"Yes/no","factoid":"Factoid","list":"List","summary":"Summary"
})

x = np.arange(len(rho_plot))
w = 0.36
fig, ax = plt.subplots(figsize=(8.0, 5.0))
ax.bar(x - w/2, rho_plot["raw_spearman_rho"], width=w, label="Validation")
ax.bar(x + w/2, rho_plot["locked_test_raw_spearman_rho"], width=w, label="Locked test")
ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(rho_plot["task_label"])
ax.set_ylabel("Spearman ρ: raw Ridge vs task score")
ax.set_title("Task-specific confidence discrimination")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "Figure5_confidence_discrimination_transport.png",
            dpi=FIG_DPI, bbox_inches="tight")
plt.show()

## 22. Figure 6 — Locked-test isotonic calibration diagnostics

In [ ]:
iso = confidence_test[confidence_test["method"] == "task_ridge_isotonic"].copy()
iso["task_label"] = iso["task"].map({
    "yesno":"Yes/no","factoid":"Factoid","list":"List","summary":"Summary"
})

fig, ax = plt.subplots(figsize=(8.0, 5.0))
ax.scatter(iso["mean_confidence"], iso["mean_observed_task_score"], s=90)
for row in iso.itertuples(index=False):
    ax.annotate(row.task_label, (row.mean_confidence, row.mean_observed_task_score),
                textcoords="offset points", xytext=(5,5))
ax.plot([0,1],[0,1], linewidth=1)
ax.set_xlim(0,1)
ax.set_ylim(0,1)
ax.set_xlabel("Mean calibrated expected task score")
ax.set_ylabel("Mean observed task score")
ax.set_title("Locked-test task-level calibration overview")
fig.tight_layout()
fig.savefig(FIG_DIR / "Figure6_locked_test_calibration_overview.png",
            dpi=FIG_DPI, bbox_inches="tight")
plt.show()

# Part VII — Main and supplementary tables

## 23. Main-table candidates

In [ ]:
main_table_1 = task_transport[[
    "task","validation_task_score","locked_test_task_score","delta"
]].copy()

main_table_2 = retrieval_master.copy()

main_table_3 = policy_master[[
    "policy","split","coverage","abstention_rate",
    "full_macro_task_score","selective_macro_task_score","macro_task_score_gain"
]].copy()

main_table_4 = confidence_transport[[
    "task",
    "raw_spearman_rho","raw_spearman_p",
    "locked_test_raw_spearman_rho","locked_test_raw_spearman_p",
    "locked_test_isotonic_ece",
    "calibration_bias","calibration_slope",
]].copy()

main_sheets = OrderedDict([
    ("Task performance", main_table_1),
    ("Retrieval", main_table_2),
    ("Selective policies", main_table_3),
    ("Confidence", main_table_4),
])

MAIN_XLSX = EXPORT_DIR / "manuscript_main_tables.xlsx"
save_workbook(main_sheets, MAIN_XLSX)
print("Saved:", MAIN_XLSX)

## 24. Supplementary tables

In [ ]:
# S1 — integrity
S1 = artifact_inventory.copy()

# S2 — answer quality detail
S2 = answer_master.copy()

# S3 — task contribution to transport
S3 = task_transport.copy()

# S4 — retrieval
S4 = retrieval_master.copy()

# S5 — locked-test grounding
S5 = grounding_test.copy()

# S6 — confidence/calibration
S6 = confidence_transport.copy()

# S7 — policy by task
S7 = policy_task_test.copy()

# S8 — policy bootstrap
S8 = policy_bootstrap_master.copy()

# S9 — validation vs test final comparison
S9 = val_test_comparison.copy()

# S10 — list evaluator audit
S10 = list_metric_audit.copy()

supp_sheets = OrderedDict([
    ("S1 Integrity", S1),
    ("S2 Answer quality", S2),
    ("S3 Task transport", S3),
    ("S4 Retrieval", S4),
    ("S5 Grounding", S5),
    ("S6 Confidence", S6),
    ("S7 Policy by task", S7),
    ("S8 Policy bootstrap", S8),
    ("S9 Val vs test", S9),
    ("S10 List audit", S10),
])

SUPP_XLSX = EXPORT_DIR / "manuscript_supplementary_tables.xlsx"
save_workbook(supp_sheets, SUPP_XLSX)
print("Saved:", SUPP_XLSX)

for name, df in supp_sheets.items():
    safe = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    df.to_csv(SUPP_DIR / f"{safe}.csv", index=False)

# Part VIII — Single source of truth for manuscript writing

## 25. Build `manuscript_final_numbers.json`

In [ ]:
manuscript_numbers = {
    "dataset": {
        "name": "BioASQ14b",
        "validation_n": 862,
        "locked_test_n": 860,
        "locked_test_design": "one-time locked internal hold-out",
    },
    "frozen_pipeline": {
        "retrieval": "BM25 + MedCPT -> RRF -> MedCPT Cross-Encoder",
        "evidence_strategy": "document_top5",
        "generator": "Qwen/Qwen3-4B-Instruct-2507",
        "grounding": "Phase 4C v2, fixed MedCPT Top-3, rank-1 NLI primary",
        "confidence_ranking": "task_ridge_raw",
        "reported_confidence": "task_ridge_isotonic",
    },
    "validation": {
        "answer_quality": validation_answer,
        "retrieval": validation_retrieval,
        "primary_policy": validation_policy_a,
        "secondary_policy": validation_policy_b,
        "policy_bootstrap": validation_policy_bootstrap.to_dict(orient="records"),
    },
    "locked_test": {
        "answer_quality": test_answer_dict,
        "retrieval": retrieval_test_summary,
        "confidence": confidence_test.to_dict(orient="records"),
        "grounding": grounding_test.to_dict(orient="records"),
        "primary_policy": phase5["primary_policy"],
        "secondary_policy": phase5["secondary_policy"],
        "policy_bootstrap": policy_boot_test.to_dict(orient="records"),
    },
    "validation_to_test": {
        "task_transport": task_transport.to_dict(orient="records"),
        "retrieval_transport": retrieval_master.to_dict(orient="records"),
        "policy_transport": policy_master.to_dict(orient="records"),
    },
    "integrity": {
        "phase5_final_report_sha256": sha256_file(PHASE5_FINAL_REPORT),
        "phase5_marker_status": marker["status"],
        "post_hoc_test_tuning_performed": False,
        "locked_test_used_for_model_selection": False,
        "locked_test_used_for_coverage_selection": False,
        "locked_test_used_for_threshold_tuning": False,
    },
    "important_interpretation": {
        "confidence_definition": (
            "For factoid, list, and summary, calibrated confidence is an expected benchmark "
            "task score, not a probability of clinical truth."
        ),
        "list_locked_test_correlation": (
            "The positive locked-test list correlation is post-hoc/exploratory and does not "
            "change the frozen validation-based ineligibility decision."
        ),
        "locked_test_scope": (
            "The locked test is an internal hold-out, not external validation."
        ),
    },
}

MANUSCRIPT_JSON = EXPORT_DIR / "manuscript_final_numbers.json"
save_json(manuscript_numbers, MANUSCRIPT_JSON)

print("Saved:", MANUSCRIPT_JSON)
print("SHA256:", sha256_file(MANUSCRIPT_JSON))

## 26. Manuscript-ready key results paragraph numbers

In [ ]:
pA = phase5["primary_policy"]
pB = phase5["secondary_policy"]

primary_boot = policy_boot_test[policy_boot_test["policy"] == PRIMARY_POLICY].iloc[0]
secondary_boot = policy_boot_test[policy_boot_test["policy"] == SECONDARY_POLICY].iloc[0]

results_text = f"""
ONE-TIME LOCKED INTERNAL TEST — KEY RESULTS

The frozen reranking pipeline achieved recall@10={retrieval_test_summary['recall@10']:.4f},
MAP@10={retrieval_test_summary['MAP@10']:.4f}, MRR@10={retrieval_test_summary['MRR@10']:.4f},
and nDCG@10={retrieval_test_summary['nDCG@10']:.4f} on the one-time locked internal test.

The full-answering system obtained MacroTaskScore={test_answer_dict['macro_task_score']:.4f}.
Task-specific locked-test scores were yes/no accuracy={test_answer_dict['yesno_accuracy']:.4f},
factoid MRR={test_answer_dict['factoid_mrr']:.4f}, list F1={test_answer_dict['list_f1']:.4f},
and summary ROUGE-L={test_answer_dict['summary_rougeL_f']:.4f}.

The prespecified primary policy retained {pA['overall_question_coverage']*100:.1f}% overall
coverage and improved MacroTaskScore from {pA['full_macro_task_score']:.4f} to
{pA['selective_macro_task_score']:.4f}, a gain of {pA['macro_task_score_gain']:.4f}
(95% bootstrap CI {primary_boot['ci95_low']:.4f} to {primary_boot['ci95_high']:.4f}).

The prespecified secondary policy retained {pB['overall_question_coverage']*100:.1f}% overall
coverage and improved MacroTaskScore from {pB['full_macro_task_score']:.4f} to
{pB['selective_macro_task_score']:.4f}, a gain of {pB['macro_task_score_gain']:.4f}
(95% bootstrap CI {secondary_boot['ci95_low']:.4f} to {secondary_boot['ci95_high']:.4f}).

No test-set tuning, model selection, coverage selection, or threshold selection was performed.
"""

RESULTS_TXT = EXPORT_DIR / "manuscript_key_results.txt"
RESULTS_TXT.write_text(results_text.strip() + "\n", encoding="utf-8")
print(results_text)

# Part IX — Final methods / results / discussion map

## 27. Recommended manuscript structure

In [ ]:
manuscript_map = pd.DataFrame([
    ["Methods", "Dataset and duplicate-safe split", "Phase 1–2", "Dataset integrity; 4007/862/860 split"],
    ["Methods", "Closed-corpus retrieval", "Phase 3", "BM25 + MedCPT -> RRF -> MedCPT CE"],
    ["Methods", "Generation and evidence pruning", "Phase 4A–4B", "Qwen3-4B; document_top5"],
    ["Methods", "Grounding assessment", "Phase 4C v2", "MedCPT fixed Top-3 + biomedical NLI"],
    ["Methods", "Task-specific confidence", "Phase 4D v2", "Per-task Ridge + nested OOF isotonic"],
    ["Methods", "Selective answering", "Phase 4E", "Prespecified eligibility and 80% coverage"],
    ["Methods", "Locked evaluation protocol", "Phase 5", "Stage A label-blind freeze then Stage B gold scoring"],
    ["Results", "Retrieval", "Phase 3 + 5", "Validation and locked-test retrieval transport"],
    ["Results", "Answer quality", "Phase 4B + 5", "Task-level and MacroTaskScore"],
    ["Results", "Grounding", "Phase 4C + 5", "Grounding does not equal correctness"],
    ["Results", "Confidence", "Phase 4D + 5", "Calibration does not equal discrimination"],
    ["Results", "Selective answering", "Phase 4E + 5", "Policy A/B transport"],
    ["Discussion", "Task heterogeneity", "Phase 4D–5", "Summary/factoid useful; yes/no not; list exploratory"],
    ["Discussion", "Transport limits", "Phase 5", "List-answer performance drop despite stable retrieval"],
    ["Discussion", "Clinical interpretation", "All", "Benchmark expected score ≠ clinical correctness probability"],
    ["Limitations", "Internal hold-out", "Phase 5", "Not external validation"],
    ["Limitations", "Automatic grounding", "Phase 4C", "NLI signal uncalibrated for clinical truth"],
], columns=["manuscript_section","subsection","source_phase","content"])

manuscript_map.to_csv(EXPORT_DIR / "manuscript_structure_map.csv", index=False)
display(manuscript_map)

# Part X — Final audit and freeze

## 28. Final consistency checks

In [ ]:
# Policy gains should match Phase 5 final report.
pA_row = policy_summary_test[policy_summary_test["policy"] == PRIMARY_POLICY].iloc[0]
pB_row = policy_summary_test[policy_summary_test["policy"] == SECONDARY_POLICY].iloc[0]

assert abs(
    float(pA_row["macro_task_score_gain"])
    - float(phase5["primary_policy"]["macro_task_score_gain"])
) < 1e-12

assert abs(
    float(pB_row["macro_task_score_gain"])
    - float(phase5["secondary_policy"]["macro_task_score_gain"])
) < 1e-12

# Validation frozen policies remain unchanged.
assert abs(
    float(validation_policy_a["macro_task_score_gain"]) - 0.0020873470636947333
) < 1e-12
assert abs(
    float(validation_policy_b["macro_task_score_gain"]) - 0.011147804580034615
) < 1e-12

# Final test hash is still exactly the completion-marker hash.
assert sha256_file(PHASE5_FINAL_REPORT) == marker["final_report_sha256"]

print("PASS — final numeric consistency checks.")

## 29. Freeze Phase 6 manuscript package

In [ ]:
phase6_outputs = sorted([
    p for p in PHASE6_DIR.rglob("*")
    if p.is_file()
])

phase6_manifest = {
    "dataset": "BioASQ14b",
    "phase": "Phase 6 Final Analysis & Manuscript Package",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "phase5_status": marker["status"],
    "phase5_final_report_sha256": sha256_file(PHASE5_FINAL_REPORT),
    "locked_test_reopened": False,
    "model_training_performed": False,
    "model_tuning_performed": False,
    "threshold_tuning_performed": False,
    "coverage_tuning_performed": False,
    "policy_changed_after_test": False,
    "files": {
        str(p.relative_to(PHASE6_DIR)): sha256_file(p)
        for p in phase6_outputs
        if p.name != "phase6_freeze_manifest.json"
    },
}

PHASE6_MANIFEST = PHASE6_DIR / "phase6_freeze_manifest.json"
save_json(phase6_manifest, PHASE6_MANIFEST)

print("PHASE 6 FROZEN.")
print("Manifest:", PHASE6_MANIFEST)
print("Manifest SHA256:", sha256_file(PHASE6_MANIFEST))

# Final outputs to review

Please send the following after the notebook completes:

1. `Table_answer_quality_validation_vs_locked_test.csv`
2. `Table_task_contribution_to_macro_transport.csv`
3. `Table_retrieval_validation_vs_locked_test.csv`
4. `Table_selective_policy_validation_vs_locked_test.csv`
5. `Table_confidence_transport.csv`
6. `list_evaluator_consistency_audit.json`
7. Figures 1–6
8. `manuscript_final_numbers.json`
9. `manuscript_main_tables.xlsx`
10. `manuscript_supplementary_tables.xlsx`
11. `phase6_freeze_manifest.json`

After Phase 6, the experimental workflow is closed. The next step is manuscript drafting,
not additional model tuning.